# 11_flight_arrivals_api_prototype

Prototype notebook for tomorrow's arrivals using AeroDataBox via RapidAPI.

In [ ]:
import os
import pandas as pd
import requests
from datetime import datetime, timedelta

# Set these in your environment before running:
# RAPIDAPI_KEY

In [ ]:
import pandas as pd
import requests
from datetime import datetime, timedelta

RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY", "ADD_YOUR_KEY_HERE")
RAPIDAPI_HOST = "aerodatabox.p.rapidapi.com"

airport_codes = {
    "Berlin": "BER",
    "Hamburg": "HAM",
    "Munich": "MUC"
}

def get_tomorrow_dates():
    tomorrow = datetime.now() + timedelta(days=1)
    tomorrow_str = tomorrow.strftime("%Y-%m-%d")
    from_local = f"{tomorrow_str}T00:00"
    to_local = f"{tomorrow_str}T23:59"
    return from_local, to_local

def get_tomorrows_arrivals(airport_codes, api_key):
    from_local, to_local = get_tomorrow_dates()

    headers = {
        "X-RapidAPI-Key": api_key,
        "X-RapidAPI-Host": RAPIDAPI_HOST
    }

    all_flights = []

    for city, airport_code in airport_codes.items():
        url = f"https://{RAPIDAPI_HOST}/flights/airports/iata/{airport_code}/{from_local}/{to_local}"

        params = {
            "direction": "Arrival",
            "withCancelled": "false",
            "withCodeshared": "true",
            "withCargo": "false",
            "withPrivate": "false",
            "withLeg": "true"
        }

        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()

        arrivals = data.get("arrivals", [])

        for flight in arrivals:
            arrival = flight.get("arrival", {})
            departure = flight.get("departure", {})
            airline = flight.get("airline", {})

            all_flights.append({
                "city": city,
                "airport_iata": airport_code,
                "flight_number": flight.get("number"),
                "airline_name": airline.get("name"),
                "departure_airport": (departure.get("airport") or {}).get("name"),
                "arrival_scheduled_time_local": (arrival.get("scheduledTime") or {}).get("local"),
                "arrival_terminal": arrival.get("terminal"),
                "arrival_gate": arrival.get("gate"),
                "status": flight.get("status")
            })

    flights_df = pd.DataFrame(all_flights)
    flights_df["arrival_scheduled_time_local"] = pd.to_datetime(
        flights_df["arrival_scheduled_time_local"], errors="coerce"
    )

    return flights_df

In [ ]:
city_lookup_query = """
SELECT city_id, city
FROM cities
"""
cities_lookup = pd.read_sql(city_lookup_query, engine)

city_flights_df = get_tomorrows_arrivals(airport_codes, RAPIDAPI_KEY)
city_flights_df = city_flights_df.merge(cities_lookup, on="city", how="left")
city_flights_df.to_sql("city_flights", con=engine, if_exists="replace", index=False)

In [ ]:
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY", "ADD_YOUR_KEY_HERE")